# Baseline Audit (The Starting Point)

In [0]:
%sql
-- Current state of the table
SELECT 'Initial State' AS event, COUNT(*) AS row_count, AVG(price_rub) AS avg_price
FROM vstone_catalog.silver.listings_silver_merged;

# Transactional Update (Atomicity & Durability)

In [0]:
%sql
-- ATOMICITY: Increase price by 10% for all 'toyota' cars
-- If this query fails midway, no row's price will be changed.
-- UPDATE operation: Increasing price_rub and price_usd by 10% for 'toyota' cars
UPDATE vstone_catalog.silver.listings_silver_merged
SET price_rub = price_rub * 1.10,
    price_usd = (price_rub * 1.10) / 82.5
WHERE brand = 'toyota';

-- Verify result: Display updated price_rub and price_usd for 'toyota' cars
SELECT brand, model, price_rub, price_usd 
FROM vstone_catalog.silver.listings_silver_merged 
WHERE brand = 'toyota' 
LIMIT 5;

# Lineage & History (The Audit Trail)

In [0]:
%sql
-- DURABILITY: See when and what changes were made
-- This will show the 'UPDATE' operation we just performed
DESCRIBE HISTORY vstone_catalog.silver.listings_silver_merged;

# Delta Time Travel

In [0]:
%sql
-- STEP 4: TIME TRAVEL QUERY
-- Viewing data from Version 0 (state before the update).
-- This shows that our "Old Price" is still safely stored in log files.

SELECT 
    'Old Price' AS version,
    brand,
    AVG(price_rub) AS avg_price
FROM vstone_catalog.silver.listings_silver_merged VERSION AS OF 13
WHERE brand = 'toyota'
GROUP BY brand

UNION ALL

SELECT 
    'New Price' AS version,
    brand,
    AVG(price_rub) AS avg_price
FROM vstone_catalog.silver.listings_silver_merged VERSION AS OF 15
WHERE brand = 'toyota'
GROUP BY brand;